In [1]:
import pandas as pd


In [5]:
results = pd.read_csv(
    "results/baseline_ragas_results.csv"
)

results[["case_id", "faithfulness", "context_precision", "response_relevancy", "factual_correctness"]]

,case_id,faithfulness,context_precision,response_relevancy,factual_correctness
0,text_simple_01,0.481481,0.000,0.734717,0.80
1,text_complex_01,0.342105,0.325,0.663383,0.55
2,numeric_simple_01,0.233333,0.000,0.753206,0.15
3,numeric_complex_01,0.190476,0.000,0.774549,0.00
4,numeric_complex_02,0.200000,0.000,0.743218,0.15
5,comparison_01,0.000000,0.250,0.851608,0.15
6,robustness_noisy_01,0.631579,0.000,NaN,0.48
7,out_of_scope_01,0.000000,0.000,0.000000,0.53


Le principal problème se trouve au niveau du retriever, pas uniquement du LLM :

- Context Precision est nulle pour six questions sur huit.
- Les réponses sont souvent pertinentes dans leur formulation.
- Mais elles ne disposent pas des bonnes données, ce qui réduit Faithfulness et Factual Correctness.

C’est particulièrement visible pour les questions numériques :

- numeric_complex_01 obtient 0 en exactitude factuelle ;
- numeric_complex_02 obtient 0,15 ;
- comparison_01 obtient une pertinence élevée (0,85), mais une fidélité nulle et une exactitude de seulement 0,15.

Le LLM comprend donc la demande, mais FAISS ne lui fournit probablement pas toutes les lignes nécessaires pour filtrer, classer ou comparer.

In [3]:
for _, row in results.iterrows():
    print("Question :")
    print(row["question"])

    print("\nRéponse :")
    print(row["answer"])

    print("\n" + "-" * 80 + "\n")

Question :
Quels sont les trois joueurs les plus souvent mentionnés dans les commentaires ?

Réponse :
Ah, excellente question ! D’après les échanges récents sur Reddit (et en croisant les tendances de *shit talk*, de débats stratégiques et de rivalités), voici les **trois joueurs les plus souvent cités** dans les commentaires animés, avec une nuance sur leur impact réel vs. leur image médiatique :

---

### **1. Tyrese Haliburton (Thunder OKC)**
**Pourquoi ?**
- **Le "shit talk" roi** : Haliburton est souvent accusé (à tort ou à raison) d’être le plus vocal des joueurs NBA, avec des répliques cinglantes contre ses adversaires (ex. : *"I’m not here to play for you"* vs. les Warriors en 2023). Les fans adorent (ou détestent) son côté provocateur, surtout en playoffs.
- **Rivalité historique** : Son duel avec Jayson Tatum (Celtics) ou Devin Booker (Warriors) a alimenté les débats sur la "mentalité de leader" vs. la "mentalité de joueur". Les commentaires sur Reddit (et les réseaux sociau

# Analyse des résultats de l’évaluation RAGAS

## 1. Résultats globaux

| Métrique            | Moyenne approximative | Interprétation                                                                          |
| ------------------- | --------------------: | --------------------------------------------------------------------------------------- |
| Faithfulness        |                  0,26 | Faible : plusieurs affirmations ne sont pas soutenues par les contextes récupérés       |
| Context Precision   |                  0,07 | Très faible : les passages récupérés sont rarement adaptés à la question                |
| Response Relevancy  |                  0,65 | Correcte : les réponses restent généralement liées au sujet de la question              |
| Factual Correctness |                  0,35 | Faible : les réponses contiennent de nombreuses informations différentes des références |

Les réponses sont donc généralement liées au sujet demandé, mais elles utilisent rarement les bonnes informations et contiennent de nombreuses erreurs factuelles.

L’analyse manuelle montre également que certains scores RAGAS sont trop généreux. Les métriques automatiques doivent donc être complétées par une vérification humaine des réponses.

---

## 2. Analyse par question

### 2.1 Joueurs les plus souvent mentionnés

**Question :**

> Quels sont les trois joueurs les plus souvent mentionnés dans les commentaires ?

**Réponse attendue :**

> Reggie Miller, Stephen Curry et Dwyane Wade.

**Réponse générée :**

> Tyrese Haliburton, Jayson Tatum et Devin Booker.

La réponse est incorrecte : aucun des trois joueurs attendus n’est identifié.

Elle contient également plusieurs hallucinations :

* Tyrese Haliburton est associé au Thunder alors qu’il joue aux Pacers ;
* Devin Booker est associé aux Warriors alors qu’il joue aux Suns ;
* plusieurs rivalités, citations et statistiques sont inventées.

Le score `Factual Correctness` de `0,80` est donc incohérent avec le contenu réel de la réponse. Le modèle juge a probablement été influencé par la longueur de la réponse et par la présence de nombreux termes liés à la NBA.

**Verdict manuel : échec.**

---

### 2.2 Principaux sujets de débat

**Question :**

> Quels sont les principaux sujets de débat dans les commentaires des supporters ?

La réponse retrouve certains thèmes réellement présents :

* la médiatisation de la NBA ;
* les équipes issues de petits marchés ;
* la parité entre les équipes ;
* l’intérêt du public pour une finale entre le Thunder et les Pacers.

Cependant, plusieurs thèmes importants sont absents :

* les équipes et joueurs les plus impressionnants des playoffs ;
* l’efficacité et la place historique de Reggie Miller ;
* l’avantage du terrain ;
* le classement des équipes ;
* le système de play-in.

La réponse ajoute également des thèmes secondaires ou non suffisamment soutenus par les documents :

* la manipulation des résultats ;
* le trash-talk de Tyrese Haliburton ;
* l’arbitrage ;
* différentes citations et statistiques non présentes dans les sources.

**Verdict manuel : réponse partiellement correcte, mais incomplète et fortement enrichie par des informations inventées.**

---

### 2.3 Meilleur pourcentage à trois points

**Question :**

> Quels joueurs possèdent le meilleur pourcentage de réussite à trois points ?

**Réponse attendue :**

> Alondes Williams et Skal Labissiere avec 100 %, chacun sur une seule tentative.

La réponse générée désigne notamment Jayson Tatum, Jaren Jackson Jr., Tyrese Haliburton, Devin Booker et Jalen Brunson.

Elle ne retrouve donc aucun des deux joueurs attendus et invente plusieurs pourcentages, équipes et statistiques.

Le score `Factual Correctness` de `0,15` est cohérent avec cette réponse très éloignée de la référence.

**Verdict manuel : échec complet.**

---

### 2.4 Meilleur pourcentage avec au moins 100 tentatives

**Question :**

> Parmi les joueurs ayant tenté au moins 100 tirs à trois points, lequel possède le meilleur pourcentage de réussite ?

**Réponse attendue :**

> Seth Curry avec 45,6 % sur 184 tentatives.

**Réponse générée :**

> Reggie Miller avec 43,4 %.

La réponse utilise des données historiques externes au lieu d’appliquer le filtre demandé aux données du fichier Excel.

Le score `Factual Correctness` de `0` correspond correctement à cet échec.

**Verdict manuel : échec complet.**

---

### 2.5 Cinq meilleurs marqueurs

**Question :**

> Quels sont les cinq joueurs ayant marqué le plus de points, classés du premier au cinquième ?

**Classement attendu :**

1. Shai Gilgeous-Alexander : 2 485 points ;
2. Anthony Edwards : 2 180 points ;
3. Nikola Jokić : 2 072 points ;
4. Giannis Antetokounmpo : 2 037 points ;
5. Jayson Tatum : 1 930 points.

La réponse générée propose principalement :

1. Jayson Tatum ;
2. Nikola Jokić ;
3. Giannis Antetokounmpo ;
4. Luka Dončić ;
5. Jrue Holiday.

Elle mentionne ensuite Shai Gilgeous-Alexander dans son développement, mais ne produit jamais le classement attendu. Elle confond également les points totaux et les moyennes de points par match.

La présence de quelques joueurs appartenant réellement au classement explique probablement le score `Factual Correctness` de `0,15`, mais la réponse finale reste incorrecte.

**Verdict manuel : échec.**

---

### 2.6 Comparaison des rebonds

**Question :**

> Compare le nombre total de rebonds de Shai Gilgeous-Alexander et d’Anthony Edwards.

**Réponse attendue :**

> Shai Gilgeous-Alexander totalise 380 rebonds, contre 450 pour Anthony Edwards. Anthony Edwards a donc capté 70 rebonds de plus.

La réponse générée estime environ 1 800 rebonds pour Shai Gilgeous-Alexander et 1 200 pour Anthony Edwards. Elle conclut donc, à tort, que Shai est le meilleur rebondeur.

Elle contient aussi plusieurs hallucinations :

* Shai Gilgeous-Alexander est présenté comme un pivot ;
* Anthony Edwards est également assimilé à un intérieur ;
* les totaux et pourcentages de rebonds sont inventés ;
* les moyennes par match sont très éloignées des données réelles.

Le score élevé de `Response Relevancy` (`0,85`) signifie seulement que la réponse parle bien des deux joueurs et des rebonds. Il ne mesure pas l’exactitude des nombres ou de la conclusion.

**Verdict manuel : échec complet.**

---

### 2.7 Robustesse face aux fautes

**Question :**

> c ki les joueur ki on le meilleur pourcentaj a 3 points ?

Le système comprend correctement l’intention malgré les fautes. La robustesse linguistique est donc satisfaisante.

Cependant, la réponse reprend un classement incorrect proche de celui fourni pour la question numérique simple. Elle ne cite pas Alondes Williams et Skal Labissiere.

La valeur manquante de `Response Relevancy` correspond à un échec technique du calcul de la métrique. Elle ne doit pas être interprétée comme un score nul.

**Verdict manuel :**

* compréhension de la question : réussie ;
* exactitude de la réponse : échouée.

---

### 2.8 Question hors périmètre

**Question :**

> Quel temps fera-t-il demain à Toulouse ?

Le système indique correctement que les données fournies ne contiennent pas d’informations météorologiques. Le mécanisme de refus fonctionne donc.

Cependant, la réponse continue ensuite sur plusieurs paragraphes avec des analogies entre la météo et la NBA ainsi que de nouvelles affirmations non vérifiées.

Elle aurait dû s’arrêter après une phrase, par exemple :

> Je ne dispose pas de données météorologiques permettant de répondre à cette question.

Cela explique les résultats suivants :

* `Response Relevancy = 0` : la majorité de la réponse est inutile par rapport à la question ;
* `Faithfulness = 0` : les développements supplémentaires ne sont pas soutenus par les données ;
* `Factual Correctness = 0,53` : l’idée principale correspond partiellement à la réponse de référence.

**Verdict manuel : refus fonctionnellement correct, mais réponse trop longue et contenant des hallucinations.**

---

## 3. Bilan manuel

Sur les huit réponses évaluées :

* aucune réponse métier n’est entièrement correcte ;
* une réponse est partiellement correcte : les principaux sujets de débat ;
* une réponse refuse correctement une demande hors périmètre, mais de manière trop longue ;
* six réponses sont incorrectes ;
* le système comprend correctement la question comportant des fautes, mais fournit une réponse factuellement incorrecte.

Les résultats montrent donc que la pertinence thématique est meilleure que l’exactitude factuelle. Le système comprend généralement le sujet de la demande, mais ne parvient pas à récupérer ou exploiter correctement les données nécessaires.

---

## 4. Limites identifiées

### 4.1 Récupération insuffisante

La valeur moyenne très faible de `Context Precision` indique que les chunks transmis au LLM sont rarement les plus utiles pour répondre aux questions.

Le modèle doit alors produire une réponse avec des informations incomplètes ou non pertinentes.

### 4.2 Limites de FAISS pour les calculs globaux

Les questions demandant de :

* calculer un maximum ;
* appliquer un filtre ;
* classer plusieurs joueurs ;
* comparer des valeurs ;
* compter les mentions dans l’ensemble des documents ;

nécessitent une analyse globale des données.

Une recherche sémantique limitée aux cinq chunks les plus proches ne garantit pas que toutes les lignes nécessaires seront récupérées. FAISS recherche des passages sémantiquement proches, mais ne réalise pas directement de filtre, de tri ou d’agrégation sur l’ensemble du tableau.

### 4.3 Prompt de génération trop permissif

Le modèle adopte systématiquement un style de « débat analytique » et produit des réponses longues, même lorsque la question demande un résultat numérique simple.

Ce comportement favorise :

* l’ajout d’informations inutiles ;
* l’utilisation de connaissances générales externes ;
* l’invention de statistiques ;
* les erreurs d’équipes, de saisons et de positions ;
* l’éloignement par rapport au contexte récupéré.

Le prompt devrait imposer au modèle :

1. d’utiliser uniquement les informations présentes dans le contexte ;
2. de ne jamais compléter une information manquante avec ses connaissances générales ;
3. de signaler clairement lorsque les données sont insuffisantes ;
4. de répondre directement et brièvement ;
5. de ne pas transformer systématiquement la réponse en débat.

### 4.4 Limites du modèle juge

Certains scores de `Factual Correctness` ne correspondent pas à la qualité réelle des réponses.

L’exemple le plus évident est la première question :

* réponse entièrement incorrecte ;
* `Factual Correctness = 0,80`.

Le modèle juge `ministral-3b-2512` semble donc avoir des difficultés à comparer précisément une réponse longue avec une référence courte. Les scores automatiques doivent rester accompagnés d’une vérification manuelle.

---

## 5. Pistes d’amélioration

Les principales améliorations possibles sont :

1. utiliser un traitement structuré pour les questions numériques ;
2. appliquer directement les filtres, tris et agrégations sur le DataFrame ;
3. réserver FAISS aux questions textuelles et à la recherche dans les commentaires Reddit ;
4. router les questions vers le bon type de traitement avant la génération ;
5. renforcer le prompt afin d’interdire les connaissances externes et les valeurs inventées ;
6. produire des réponses plus courtes et directement centrées sur la question ;
7. utiliser un modèle juge plus performant pour les métriques RAGAS ;
8. conserver une vérification humaine sur un échantillon de réponses.

Une architecture hybride serait plus adaptée :

* questions textuelles sur Reddit : recherche vectorielle avec FAISS ;
* questions numériques sur les joueurs : requêtes structurées sur le DataFrame ;
* génération finale : formulation concise à partir des résultats calculés.

---

## 6. Conclusion

> L’analyse manuelle révèle que les réponses restent généralement liées au sujet, mais qu’elles contiennent de nombreuses erreurs factuelles et hallucinations. Les scores automatiques, notamment l’exactitude factuelle, doivent donc être interprétés avec prudence. Les principales limites proviennent de l’utilisation d’une recherche vectorielle pour des calculs globaux et d’un prompt de génération trop permissif. Une architecture hybride combinant recherche sémantique et traitement structuré des statistiques permettrait d’améliorer fortement la fiabilité des réponses.
